
**Movie Recommender — Fine-tuned Genre Classifier + RAG (Gradio)**

A simple web app that recommends movies using a fine-tuned multi-label genre classifier (DistilBERT) and a hybrid RAG retriever (BM25 + sentence embeddings).
Built with a Gradio UI.


**Features**

*   Fine-tunes distilbert-base-uncased to predict multiple genres from text.
*   Hybrid retrieval: BM25 (keyword) + MiniLM embeddings (semantic).
*   Intent parsing for mood / actor / director / year / decade / year-range.
*   Hard year filtering (e.g., 2010–2015, 90s) + light boost for higher ratings.
*   Gradio app with share link


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/movie_app"

for folder in ["data", "models", "index", "src"]:
    os.makedirs(f"{BASE}/{folder}", exist_ok=True)

print("Project folder:", BASE)
print("Ready:", os.path.exists(BASE))

In [ ]:
!pip -q install gradio pandas numpy scikit-learn pyyaml rank-bm25 sentence-transformers transformers accelerate datasets


Model Training (Optional — Run only to retrain the model)




In [ ]:

# Fine-tuning DistilBERT for Multi-label Genre Classification

import os
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score

# Disable W&B
os.environ["WANDB_DISABLED"] = "true"

# Model configuration
MODEL_BASE = "distilbert-base-uncased"
MODEL_DIR = f"{BASE}/models/genre_clf"

os.makedirs(MODEL_DIR, exist_ok=True)

# Load tokenizer
tok = AutoTokenizer.from_pretrained(MODEL_BASE)

# Convert labels to float32 for BCEWithLogitsLoss
y_train_f = y_train.astype("float32")
y_val_f = y_val.astype("float32")

# Create Hugging Face datasets
train_ds = Dataset.from_dict({
    "text": X_train,
    "labels": [row.tolist() for row in y_train_f]
})

val_ds = Dataset.from_dict({
    "text": X_val,
    "labels": [row.tolist() for row in y_val_f]
})

# Tokenization
def tok_map(batch):
    enc = tok(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    enc["labels"] = batch["labels"]
    return enc

train_ds = train_ds.map(
    tok_map,
    batched=True,
    remove_columns=["text"]
)

val_ds = val_ds.map(
    tok_map,
    batched=True,
    remove_columns=["text"]
)

# Set PyTorch format
train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# Data collator: ensure float32 labels
def collate_float(batch):
    out = {}

    for k in batch[0]:
        if k == "labels":
            out[k] = torch.stack([
                torch.tensor(ex[k], dtype=torch.float32)
                for ex in batch
            ])
        else:
            out[k] = torch.stack([
                ex[k] if torch.is_tensor(ex[k])
                else torch.tensor(ex[k])
                for ex in batch
            ])

    return out

# Load classification model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_BASE,
    num_labels=len(genres_vocab),
    problem_type="multi_label_classification"
)

# Training configuration
args = TrainingArguments(
    output_dir=MODEL_DIR,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=2,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_float
)

# Train model
trainer.train()

# Save final model and tokenizer to Google Drive
trainer.save_model(MODEL_DIR)
tok.save_pretrained(MODEL_DIR)

# Manual evaluation
pred = trainer.predict(val_ds)

logits = pred.predictions
labels = pred.label_ids

probs = 1 / (1 + np.exp(-logits))
preds = (probs >= 0.3).astype(int)

print("F1 micro:", f1_score(
    labels, preds, average="micro", zero_division=0
))

print("F1 macro:", f1_score(
    labels, preds, average="macro", zero_division=0
))

print("Model saved successfully:", os.path.exists(
    f"{MODEL_DIR}/config.json"
))


Build Search Index (Optional — Run only to rebuild the index)





In [ ]:

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd
import os

# Prepare project directories
os.makedirs(f"{BASE}/index", exist_ok=True)
os.makedirs(f"{BASE}/data", exist_ok=True)

# Build searchable text for each movie
def row_text(r):
    parts = [str(r["title"]), str(r["overview"])]

    if isinstance(r["cast_names"], list) and r["cast_names"]:
        parts.append(" ".join(r["cast_names"]))

    if isinstance(r["genres_top"], list) and r["genres_top"]:
        parts.append(" ".join(r["genres_top"]))

    if pd.notna(r["director"]) and r["director"]:
        parts.append(str(r["director"]))

    return " | ".join(parts)

corpus = movies.apply(row_text, axis=1).tolist()

# Generate semantic embeddings
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

emb = SentenceTransformer(EMBEDDING_MODEL)

E = emb.encode(
    corpus,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Build BM25 keyword search index
bm25 = BM25Okapi([
    text.lower().split() for text in corpus
])

# Save embeddings and movie data to Google Drive
np.save(f"{BASE}/index/embeddings.npy", E)
movies.to_pickle(f"{BASE}/data/movies.pkl")

# Verify saved files
print("Movies saved:", os.path.exists(f"{BASE}/data/movies.pkl"))
print("Embeddings saved:", os.path.exists(f"{BASE}/index/embeddings.npy"))
print("Movies count:", len(movies))
print("Embeddings shape:", E.shape)
print("BM25 documents:", len(corpus))


6) Inference helpers (intent parsing + hybrid search + re-rank)

In [ ]:

import re
import json
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi


# Load saved data and models
movies = pd.read_pickle(f"{BASE}/data/movies.pkl")
E = np.load(f"{BASE}/index/embeddings.npy")

emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

bm25 = BM25Okapi([
    c.lower().split()
    for c in movies.apply(
        lambda r: f"{r['title']} | {r['overview']} | {' '.join(r['cast_names'] or [])} | {' '.join(r['genres_top'] or [])} | {r['director'] or ''}",
        axis=1
    ).tolist()
])

tok = AutoTokenizer.from_pretrained(f"{BASE}/models/genre_clf")
clf = AutoModelForSequenceClassification.from_pretrained(
    f"{BASE}/models/genre_clf"
).eval()

with open(f"{BASE}/models/genre_vocab.json", "r") as f:
    genres_vocab = json.load(f)

print("Loaded genres:", len(genres_vocab))


# Genre prediction
def predict_genres(text, thr=0.3):
    with torch.no_grad():
        enc = tok(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        )
        logits = clf(**enc).logits
        probs = torch.sigmoid(logits).squeeze(0).numpy()

    idxs = np.where(probs >= thr)[0]
    predicted = [genres_vocab[i] for i in idxs]

    # Detect explicitly requested genres
    genre_aliases = {
        "Action": ["action"],
        "Adventure": ["adventure"],
        "Animation": ["animation", "animated"],
        "Comedy": ["comedy", "funny"],
        "Crime": ["crime"],
        "Drama": ["drama"],
        "Family": ["family"],
        "Fantasy": ["fantasy"],
        "History": ["history", "historical"],
        "Horror": ["horror", "scary"],
        "Music": ["music", "musical"],
        "Mystery": ["mystery", "detective"],
        "Romance": ["romance", "romantic"],
        "Science Fiction": ["science fiction", "sci-fi", "scifi"],
        "Thriller": ["thriller", "suspense"]
    }

    query = text.lower()

    for genre, aliases in genre_aliases.items():
        for alias in aliases:
            if alias in query and genre in genres_vocab:
                if genre not in predicted:
                    predicted.append(genre)
                break

    return predicted, probs


# Query intent parsing
def parse_intent(q):
    ql = q.lower()

    # Mood
    mood = None
    for m in [
        "light", "funny", "dark", "mind-bending",
        "feel-good", "romantic", "thriller",
        "slow-burn", "family"
    ]:
        if m in ql:
            mood = m
            break

    # Person
    person = None
    m3 = re.search(
        r"(starring|with|by|directed by)\s+([A-Za-z][A-Za-z\s'.-]+)",
        ql
    )
    if m3:
        person = m3.group(2).strip()

    # Year
    year = None
    m = re.search(r"\b(19|20)\d{2}\b", ql)
    if m:
        year = int(m.group(0))

    # Decade
    decade = None
    m_dec_full = re.search(r"\b((?:19|20)\d{2})s\b", ql)

    if m_dec_full:
        base = int(m_dec_full.group(1))
        decade = (base // 10) * 10
    else:
        m_dec_short = re.search(r"\b'?\s*(\d{2})s\b", ql)
        if m_dec_short:
            d = int(m_dec_short.group(1))
            century = 1900 if d >= 50 else 2000
            decade = century + d

    # Year range
    yr_min = yr_max = None

    m_rng = re.search(
        r"\b((?:19|20)\d{2})\s*(?:-|–|—|to)\s*((?:19|20)\d{2})\b",
        ql
    )

    if m_rng:
        yr_min = int(m_rng.group(1))
        yr_max = int(m_rng.group(2))
    elif decade is not None:
        yr_min = decade
        yr_max = decade + 9

    return {
        "mood": mood,
        "year": year,
        "decade": decade,
        "year_min": yr_min,
        "year_max": yr_max,
        "person": person
    }


# Hybrid search: semantic retrieval + BM25
def hybrid_search(query, k=60, year_min=None, year_max=None):
    q_emb = emb.encode(
        [query],
        normalize_embeddings=True
    )[0]

    cos = E @ q_emb
    bm = bm25.get_scores(query.lower().split())

    bm = (bm - bm.min()) / (bm.max() - bm.min() + 1e-9)
    cos_n = (cos - cos.min()) / (cos.max() - cos.min() + 1e-9)

    base = 0.6 * cos_n + 0.4 * bm

    # Apply year filtering before selecting top results
    if year_min is not None or year_max is not None:
        years = movies["year"].fillna(0).astype(int).values

        lo = year_min if year_min is not None else years.min()
        hi = year_max if year_max is not None else years.max()

        valid_ids = np.where(
            (years >= lo) & (years <= hi)
        )[0]

        top = valid_ids[np.argsort(-base[valid_ids])[:k]]
    else:
        top = np.argsort(-base)[:k]

    return top, base[top]


# Semantic re-ranking
def rerank(ids, base_scores, facets, k=10, query=None):
    bonus = np.zeros_like(base_scores, dtype=float)

    # Semantic similarity with the original query
    if query:
        query_embedding = emb.encode(
            [query],
            normalize_embeddings=True
        )[0]

        semantic_scores = E[ids] @ query_embedding

        semantic_min = semantic_scores.min()
        semantic_max = semantic_scores.max()

        semantic_norm = (
            (semantic_scores - semantic_min) /
            (semantic_max - semantic_min + 1e-9)
        )

        bonus += 0.15 * semantic_norm

        # Topic keyword matching
        topic_keywords = {
            "space": [
                "astronaut", "astronauts", "space", "planet",
                "planets", "mars", "moon", "spacecraft",
                "spaceship", "orbit", "galaxy", "cosmic"
            ],
            "detective": [
                "detective", "investigation", "murder",
                "mystery", "crime", "police"
            ],
            "war": [
                "war", "soldier", "battle", "military",
                "army", "combat"
            ]
        }

        query_lower = query.lower()

        overview_texts = (
            movies.iloc[ids]["overview"]
            .fillna("")
            .str.lower()
        )

        for topic, keywords in topic_keywords.items():
            if any(word in query_lower for word in keywords):
                hits = overview_texts.apply(
                    lambda text: sum(
                        1 for word in keywords if word in text
                    )
                ).values

                bonus += 0.04 * np.minimum(hits, 3)

    # Genre alignment
    want_genres = set(facets.get("genres_pred", []))
    explicit_genres = set(facets.get("explicit_genres", []))

    for i, mid in enumerate(ids):
        movie_genres = set(movies.iloc[mid]["genres_top"] or [])

        if explicit_genres:
            explicit_match = len(movie_genres & explicit_genres)
            other_match = len(
                movie_genres & (want_genres - explicit_genres)
            )

            bonus[i] += 0.20 * explicit_match
            bonus[i] += 0.01 * other_match

        elif want_genres:
            bonus[i] += 0.01 * len(
                movie_genres & want_genres
            )

    # Mood matching
    mood = facets.get("mood")

    if mood:
        mood_words = {
            "light": ["fun", "light", "comedy", "feel good", "feel-good"],
            "funny": ["comedy", "funny", "humor"],
            "dark": ["dark", "noir", "crime", "horror"],
            "mind-bending": ["mind", "twist", "puzzle", "dream", "loop"],
            "feel-good": ["feel good", "heartwarming", "uplifting"],
            "romantic": ["romance", "romantic", "love"],
            "thriller": ["thriller", "suspense", "tense"],
            "slow-burn": ["slow burn", "character-driven", "atmospheric"],
            "family": ["family", "kids", "children"]
        }

        kws = mood_words.get(mood, [])

        if kws:
            texts = (
                movies.iloc[ids]["overview"]
                .str.lower()
                .fillna("")
            )

            hit = texts.apply(
                lambda t: any(w in t for w in kws)
            ).values.astype(float)

            bonus += 0.05 * hit

    # Person matching
    person = facets.get("person")

    if person:
        p = person.lower()
        mask = []

        for mid in ids:
            cast = [
                c.lower()
                for c in (movies.iloc[mid]["cast_names"] or [])
            ]

            director = (
                movies.iloc[mid]["director"] or ""
            ).lower()

            mask.append(
                1.0 if any(p in c for c in cast)
                or p in director else 0.0
            )

        bonus += 0.10 * np.array(mask)

    # Year matching
    years = (
        movies.iloc[ids]["year"]
        .fillna(0)
        .astype(int)
        .values
    )

    year = facets.get("year")
    decade = facets.get("decade")

    if year:
        bonus += 0.05 * (years == year)
    elif decade:
        bonus += 0.05 * ((years // 10 * 10) == decade)

    # Rating boost
    votes = (
        movies.iloc[ids]["vote_average"]
        .fillna(0)
        .values
    )

    votes_n = (
        (votes - votes.min()) /
        (votes.max() - votes.min() + 1e-9)
    )

    bonus += 0.05 * votes_n

    # Final ranking
    final = base_scores + bonus
    order = np.argsort(-final)
    topk = order[:k]

    return ids[topk], final[topk]


7) Gradio web app

In [ ]:


import gradio as gr
import pandas as pd


def recommend(query, k):
    if not query or not query.strip():
        return "Please describe the type of movie you are looking for."

    # Parse user intent
    facets = parse_intent(query)

    # Predict genres
    genres_pred, probs = predict_genres(query)

    # Detect explicitly requested genres
    genre_aliases = {
        "Action": ["action"],
        "Adventure": ["adventure"],
        "Animation": ["animation", "animated"],
        "Comedy": ["comedy", "funny"],
        "Crime": ["crime"],
        "Drama": ["drama"],
        "Family": ["family"],
        "Fantasy": ["fantasy"],
        "History": ["history", "historical"],
        "Horror": ["horror", "scary"],
        "Music": ["music", "musical"],
        "Mystery": ["mystery", "detective"],
        "Romance": ["romance", "romantic"],
        "Science Fiction": ["science fiction", "sci-fi", "scifi"],
        "Thriller": ["thriller", "suspense"]
    }

    query_lower = query.lower()

    explicit_genres = [
        genre
        for genre, aliases in genre_aliases.items()
        if genre in genres_vocab
        and any(alias in query_lower for alias in aliases)
    ]

    facets["explicit_genres"] = explicit_genres
    facets["genres_pred"] = genres_pred

    # Hybrid search with year filtering
    ids, base = hybrid_search(
        query,
        k=100,
        year_min=facets.get("year_min"),
        year_max=facets.get("year_max")
    )

    # Filter by explicitly requested genres
    if explicit_genres:
        valid_mask = movies.iloc[ids]["genres_top"].apply(
            lambda gs: bool(set(gs) & set(explicit_genres))
        ).values

        ids = ids[valid_mask]
        base = base[valid_mask]

    # Exclude movies with zero ratings
    valid_mask = (
        movies.iloc[ids]["vote_average"]
        .fillna(0)
        .values > 0
    )

    ids = ids[valid_mask]
    base = base[valid_mask]

    if len(ids) == 0:
        return "No matching movies found. Try a different query."

    # Rerank retrieved movies
    top_ids, final_scores = rerank(
        ids,
        base,
        facets,
        k=int(k),
        query=query
    )

    rows = []

    for mid, sc in zip(top_ids, final_scores):
        r = movies.iloc[int(mid)]

        overview = (
            str(r["overview"])
            if pd.notna(r["overview"])
            else ""
        )

        rows.append({
            "title": r["title"],
            "year": int(r["year"]) if pd.notna(r["year"]) else None,
            "genres": ", ".join(r["genres_top"]),
            "vote": float(r["vote_average"]),
            "director": r["director"] if r["director"] else "",
            "cast": ", ".join(r["cast_names"][:5])
                    if isinstance(r["cast_names"], list) else "",
            "overview": overview[:400] +
                        ("..." if len(overview) > 400 else "")
        })

    # Build recommendation output
    header = (
        f"Predicted genres: {', '.join(genres_pred) or '—'}\n"
        f"Explicit genres: {', '.join(explicit_genres) or '—'}\n"
        f"Detected intent: {facets}\n"
        f"Year range: {facets.get('year_min')} - "
        f"{facets.get('year_max')}"
    )

    out_lines = [header, ""]

    for i, r in enumerate(rows, 1):
        out_lines.append(
            f"{i}. {r['title']} ({r['year']}) "
            f"[{r['genres']}] ⭐ {r['vote']}"
        )

        if r["director"]:
            out_lines.append(f"   Director: {r['director']}")

        if r["cast"]:
            out_lines.append(f"   Cast: {r['cast']}")

        out_lines.append(f"   {r['overview']}")
        out_lines.append("")

    return "\n".join(out_lines)


# Gradio interface
demo = gr.Interface(
    fn=recommend,
    inputs=[
        gr.Textbox(
            label="Describe what you want (genre, mood, actor, or year)"
        ),
        gr.Slider(
            5,
            20,
            value=10,
            step=1,
            label="Number of recommendations"
        )
    ],
    outputs=gr.Textbox(
        lines=30,
        label="Movie Recommendations"
    ),
    title="Movie Recommender — Hybrid Retrieval & Multi-label Genre Classification",
    description=(
        "Discover movies using semantic search, BM25 keyword retrieval, "
        "and multi-label genre classification."
    )
)

demo.launch(share=True)
